# LeetCode 3490 — Count Beautiful Numbers bằng Python

- Link: https://leetcode.com/problems/count-beautiful-numbers/
- Độ khó: **Hard**
- Chủ đề: **Digit DP** với trạng thái `(số chữ số còn lại, tổng chữ số, tích chữ số)`, rút gọn trường hợp **tích bằng 0**, `F(r) − F(l − 1)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def beautifulNumbers(self, l: int, r: int) -> int:
```

## 1. Mục tiêu học

1. Đưa bài đếm trên đoạn về `F(r) − F(l − 1)` với `F(N)` = số số đẹp trong `[1, N]`.
2. Thiết kế digit DP: duyệt các chữ số của `N` từ trái, phần "đã nhỏ hơn `N`" thì các chữ số sau **tự do** ⇒ hàm `free(rem, s, p)` có nhớ.
3. Nhận ra **tích bằng 0** (có chữ số 0) thì luôn chia hết ⇒ đếm ngay `10^{rem}` cách hoàn thành.
4. Hiểu vì sao số trạng thái nhỏ: tổng ≤ 81, tích chỉ gồm thừa số 2, 3, 5, 7 ⇒ vài nghìn giá trị khác nhau.
5. Kiểm chứng bằng vét cạn trên đoạn nhỏ.

## 2. Đề bài & ví dụ

Số nguyên dương gọi là **đẹp** nếu **tích các chữ số** chia hết cho **tổng các chữ số**. Đếm số đẹp trong `[l, r]`.

**Ví dụ 1:** `l = 10, r = 20` → `2` (10 và 20: tích 0 chia hết cho mọi tổng).

**Ví dụ 2:** `l = 1, r = 15` → `10` (`1..9` vì tích = tổng; và `10`).

**Ràng buộc:** `1 ≤ l ≤ r < 10^9`.

In [1]:
examples = [(10, 20, 2), (1, 15, 10)]
for l, r, exp in examples:
    print(l, r, '-> kỳ vọng', exp)

10 20 -> kỳ vọng 2
1 15 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Tích và tổng chữ số

In [2]:
def digit_ps(x):
    p, s = 1, 0
    for ch in str(x):
        p *= int(ch)
        s += int(ch)
    return p, s

def beautiful(x):
    p, s = digit_ps(x)
    return p % s == 0

print([(x, digit_ps(x), beautiful(x)) for x in [10, 12, 22, 123, 36]])

[(10, (0, 1), True), (12, (2, 3), False), (22, (4, 4), True), (123, (6, 6), True), (36, (18, 9), True)]


### 3.2. Đếm trên đoạn bằng hiệu tiền tố

`count(l, r) = F(r) − F(l − 1)`.

### 3.3. "Tách" số theo tiền tố của `N`

Mọi `x < N` (cùng số chữ số) có một vị trí đầu tiên `i` mà chữ số của `x` **nhỏ hơn** chữ số của `N`; từ đó trở đi các chữ số tự do. Số có ít chữ số hơn `N` thì hoàn toàn tự do (chữ số đầu khác 0).

### 3.4. Số lượng tích khác nhau

Tích của tối đa 9 chữ số 1..9 có dạng `2^a 3^b 5^c 7^d` ⇒ số giá trị khác nhau nhỏ.

In [3]:
prods = {1}
for _ in range(9):
    prods = prods | {p * d for p in prods for d in range(1, 10)}
print('số tích khác nhau (≤ 9 chữ số, không có 0):', len(prods))

số tích khác nhau (≤ 9 chữ số, không có 0): 3025


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với số có **chữ số 0**, tích = 0 ⇒ luôn đẹp. Với số không có 0, cần `tích % tổng == 0`.

Tính tay ví dụ 2 (`[1, 15]`): `1..9`: tích = tổng ⇒ đẹp (9 số). `10`: có 0 ⇒ đẹp. `11`: `1 % 2` ≠ 0. `12`: `2 % 3`. `13`: `3 % 4`. `14`: `4 % 5`. `15`: `5 % 6`. ⇒ 10 số.

Trạng thái khi đã "tự do": còn `rem` chữ số, tổng hiện tại `s`, tích hiện tại `p`. Kết quả chỉ phụ thuộc `(rem, s, p)` ⇒ ghi nhớ.

In [4]:
print([x for x in range(1, 16) if beautiful(x)])

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 5. Cách 1 — Brute Force: kiểm tra từng số

In [5]:
def count_brute(l, r):
    return sum(1 for x in range(l, r + 1) if beautiful(x))

for l, r, exp in examples:
    print(count_brute(l, r), '| kỳ vọng', exp)

2 | kỳ vọng 2
10 | kỳ vọng 10


**Độ phức tạp:** `O((r − l) · log r)` — tới `10^9` số: không khả thi.

**Nút thắt:** nhiều số có cùng "phần đã chọn" `(s, p)` và cùng số chữ số còn lại ⇒ tương lai giống nhau.

## 6. Ý tưởng thuật toán / bước đột phá

`free(rem, s, p)` = số cách điền `rem` chữ số tự do (0..9) sao cho số cuối cùng đẹp, biết tổng `s` (> 0) và tích `p` của phần trước:

- `p = 0` ⇒ `10^{rem}` (mọi cách đều đẹp).
- `rem = 0` ⇒ `1` nếu `p % s == 0`, ngược lại `0`.
- Ngược lại `Σ_{d=0}^{9} free(rem − 1, s + d, p · d)`.

`F(N)`:

1. Số có `L < m` chữ số: chữ số đầu `d ∈ 1..9`, cộng `free(L − 1, d, d)`.
2. Số có đúng `m` chữ số `< N`: tại vị trí `i`, chọn `d` nhỏ hơn `D[i]` (vị trí đầu `d ≥ 1`), cộng `free(m − i − 1, s + d, p · d)` với `(s, p)` của tiền tố `D[0..i−1]`.
3. Cộng 1 nếu chính `N` đẹp.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Digit DP "khuôn chuẩn" với `tight` và `started`

Viết theo khuôn chung `go(i, s, p, tight, started)`; không tối ưu riêng trường hợp tích 0 (để thấy sự khác biệt về số trạng thái).

In [6]:
from functools import lru_cache

def F_template(N):
    if N <= 0:
        return 0
    D = list(map(int, str(N)))
    m = len(D)

    @lru_cache(maxsize=None)
    def go(i, s, p, tight, started):
        if i == m:
            return 1 if started and p % s == 0 else 0
        hi = D[i] if tight else 9
        total = 0
        for d in range(hi + 1):
            nt = tight and d == hi
            if not started and d == 0:
                total += go(i + 1, 0, 1, nt, False)
            else:
                total += go(i + 1, s + d, p * d, nt, True)
        return total

    res = go(0, 0, 1, True, False)
    return res, go.cache_info().currsize

def count_template(l, r):
    return F_template(r)[0] - F_template(l - 1)[0] if l > 1 else F_template(r)[0]

for l, r, exp in examples:
    print(count_template(l, r), '| kỳ vọng', exp)

2 | kỳ vọng 2
10 | kỳ vọng 10


### 7.2. Cách 3 — Hàm `free` + duyệt tiền tố, rút gọn tích 0 (lời giải chính)

In [7]:
@lru_cache(maxsize=None)
def free(rem, s, p):
    if p == 0:
        return 10 ** rem
    if rem == 0:
        return 1 if p % s == 0 else 0
    return sum(free(rem - 1, s + d, p * d) for d in range(10))

def F(N):
    if N <= 0:
        return 0
    D = list(map(int, str(N)))
    m = len(D)
    total = 0
    for L in range(1, m):                      # ít chữ số hơn N
        for d in range(1, 10):
            total += free(L - 1, d, d)
    s, p = 0, 1
    for i, di in enumerate(D):                 # đúng m chữ số, nhỏ hơn N
        for d in range(1 if i == 0 else 0, di):
            total += free(m - i - 1, s + d, p * d)
        s += di
        p *= di
    return total + (1 if p % s == 0 else 0)    # chính N

def count_fast(l, r):
    return F(r) - F(l - 1)

for l, r, exp in examples:
    print(count_fast(l, r), '| kỳ vọng', exp)

2 | kỳ vọng 2
10 | kỳ vọng 10


Ghi nhớ `free` dùng chung cho mọi lần gọi `F` (cả `F(r)` và `F(l − 1)`), vì nó không phụ thuộc `N`.

### 7.3. Cách 4 — Chuẩn hóa tích thành bộ số mũ `(a, b, c, d)`

Thay `p` bằng bộ số mũ của 2, 3, 5, 7 (và cờ "có 0"). Tương đương về toán học, giúp hiểu vì sao số trạng thái nhỏ; kiểm tra chia hết bằng cách dựng lại tích.

In [8]:
EXP = {0: None, 1: (0, 0, 0, 0), 2: (1, 0, 0, 0), 3: (0, 1, 0, 0), 4: (2, 0, 0, 0), 5: (0, 0, 1, 0),
       6: (1, 1, 0, 0), 7: (0, 0, 0, 1), 8: (3, 0, 0, 0), 9: (0, 2, 0, 0)}

@lru_cache(maxsize=None)
def free_exp(rem, s, e):
    if e is None:                              # tích đã bằng 0
        return 10 ** rem
    if rem == 0:
        p = 2 ** e[0] * 3 ** e[1] * 5 ** e[2] * 7 ** e[3]
        return 1 if p % s == 0 else 0
    total = 0
    for d in range(10):
        ed = EXP[d]
        ne = None if ed is None else tuple(x + y for x, y in zip(e, ed))
        total += free_exp(rem - 1, s + d, ne)
    return total

def F_exp(N):
    if N <= 0:
        return 0
    D = list(map(int, str(N)))
    m = len(D)
    total = sum(free_exp(L - 1, d, EXP[d]) for L in range(1, m) for d in range(1, 10))
    s, e = 0, (0, 0, 0, 0)
    for i, di in enumerate(D):
        for d in range(1 if i == 0 else 0, di):
            ed = EXP[d]
            total += free_exp(m - i - 1, s + d, None if (e is None or ed is None) else tuple(x + y for x, y in zip(e, ed)))
        s += di
        e = None if (e is None or EXP[di] is None) else tuple(x + y for x, y in zip(e, EXP[di]))
    if e is None:
        return total + 1
    p = 2 ** e[0] * 3 ** e[1] * 5 ** e[2] * 7 ** e[3]
    return total + (1 if p % s == 0 else 0)

print([F_exp(r) - F_exp(l - 1) for l, r, _ in examples])

[2, 10]


## 8. TRACE từng bước

Trace cách 3 cho `F(234)`: in đóng góp của từng nhóm (số ít chữ số hơn, rồi từng vị trí bám tiền tố).

In [9]:
def trace_F(N):
    D = list(map(int, str(N)))
    m = len(D)
    total = 0
    for L in range(1, m):
        part = sum(free(L - 1, d, d) for d in range(1, 10))
        total += part
        print(f"số có {L} chữ số: {part}")
    s, p = 0, 1
    for i, di in enumerate(D):
        part = 0
        for d in range(1 if i == 0 else 0, di):
            part += free(m - i - 1, s + d, p * d)
        total += part
        print(f"tiền tố {D[:i]} rồi chữ số < {di} ở vị trí {i}, {m - i - 1} chữ số tự do: {part}")
        s += di; p *= di
    last = 1 if p % s == 0 else 0
    print(f"chính N = {N}: tích {p}, tổng {s} -> {'đẹp' if last else 'không đẹp'}")
    print('F =', total + last, '| vét cạn:', count_brute(1, N))

trace_F(234)

số có 1 chữ số: 9
số có 2 chữ số: 15
tiền tố [] rồi chữ số < 2 ở vị trí 0, 2 chữ số tự do: 31
tiền tố [2] rồi chữ số < 3 ở vị trí 1, 1 chữ số tự do: 14
tiền tố [2, 3] rồi chữ số < 4 ở vị trí 2, 0 chữ số tự do: 2
chính N = 234: tích 24, tổng 9 -> không đẹp
F = 71 | vét cạn: 71


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def beautifulNumbers(self, l: int, r: int) -> int:
        @lru_cache(maxsize=None)
        def free(rem, s, p):
            if p == 0:
                return 10 ** rem
            if rem == 0:
                return 1 if p % s == 0 else 0
            return sum(free(rem - 1, s + d, p * d) for d in range(10))

        def F(N):
            if N <= 0:
                return 0
            D = list(map(int, str(N)))
            m = len(D)
            total = 0
            for L in range(1, m):
                for d in range(1, 10):
                    total += free(L - 1, d, d)
            s, p = 0, 1
            for i, di in enumerate(D):
                for d in range(1 if i == 0 else 0, di):
                    total += free(m - i - 1, s + d, p * d)
                s += di
                p *= di
            return total + (1 if p % s == 0 else 0)

        return F(r) - F(l - 1)

for l, r, exp in examples:
    got = Solution().beautifulNumbers(l, r)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
10 OK


## 10. Vì sao thuật toán đúng

1. **Phân hoạch `[1, N]`:** mỗi `x ∈ [1, N]` thuộc đúng một nhóm: ít chữ số hơn `N` (theo số chữ số và chữ số đầu), hoặc cùng số chữ số và lần đầu nhỏ hơn `N` tại vị trí `i` (theo `i` và chữ số tại đó), hoặc bằng `N`.
2. **`free` đúng:** số cách hoàn thành chỉ phụ thuộc số chữ số còn lại và `(tổng, tích)` hiện tại. Tích 0 thì tích cuối cùng bằng 0, chia hết cho mọi tổng dương ⇒ `10^{rem}`.
3. **Tổng dương:** mọi số được đếm có chữ số đầu ≥ 1 ⇒ `s > 0`, không chia cho 0.
4. **Hiệu tiền tố** cho đúng số lượng trong `[l, r]`.

## 11. Phân tích Time / Space Complexity

Gọi `m ≤ 9` số chữ số, `S ≤ 81` tổng tối đa, `P` số tích khác nhau (vài nghìn).

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt | `O((r − l) m)` | `O(1)` |
| 2. Khuôn chuẩn | `O(trạng thái · 10)`, cận trên ~ `m · S · P · 4` nhưng thực tế đạt được ít hơn nhiều | như trạng thái |
| 3. `free` + tích 0 | ~ số trạng thái `(rem, s, p)` đạt được × 10 (đo ở ô dưới: vài chục nghìn) | như trạng thái |
| 4. Bộ số mũ | như 3 | như 3 |

In [11]:
res, states = F_template(10**9 - 1)
free.cache_clear()
F(10**9 - 1)
print('khuôn chuẩn: số trạng thái =', states, '| cách 3: số trạng thái free =', free.cache_info().currsize)

khuôn chuẩn: số trạng thái = 41857 | cách 3: số trạng thái free = 41838


Nhận xét trung thực: hai cách có **số trạng thái gần bằng nhau** (vì trạng thái "tight" chỉ có `m` cái và rút gọn tích 0 chỉ cắt các nhánh vốn đã ngắn). Ưu điểm thật của cách 3 là: (1) bảng nhớ `free` **không phụ thuộc `N`** nên dùng chung cho `F(r)` và `F(l − 1)` (và cho nhiều truy vấn), (2) không cần cờ `tight`/`started` nên khó sai hơn. Cả hai đều chạy trong vài phần trăm giây với `r < 10^9`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'template': count_template, 'fast': count_fast, 'exp': lambda l, r: F_exp(r) - F_exp(l - 1),
         'submit': lambda l, r: Solution().beautifulNumbers(l, r)}
edge = [(1, 1, 1), (9, 11, 2), (100, 100, 1), (11, 11, 0), (22, 22, 1), (21, 21, 0), (123, 123, 1)]
for l, r, exp in examples + edge:
    assert count_brute(l, r) == exp, (l, r)
    for name, f in impls.items():
        assert f(l, r) == exp, (name, l, r)
print('Ví dụ đề và ca biên: tất cả đúng. (22: tích 4 = tổng 4; 123: tích 6 = tổng 6)')

Ví dụ đề và ca biên: tất cả đúng. (22: tích 4 = tổng 4; 123: tích 6 = tổng 6)


### 12.2. Đối chiếu với vét cạn

In [13]:
import random
random.seed(3490)
cnt = 0
for _ in range(200):
    l = random.randint(1, 200000)
    r = l + random.randint(0, 3000)
    exp = count_brute(l, r)
    for name, f in impls.items():
        assert f(l, r) == exp, (name, l, r)
    cnt += 1
assert count_fast(1, 300000) == count_brute(1, 300000)
print('Khớp vét cạn trên', cnt, 'đoạn ngẫu nhiên và trên [1, 300000].')

Khớp vét cạn trên 200 đoạn ngẫu nhiên và trên [1, 300000].


### 12.3. Ca lớn (`r` gần `10^9`)

In [14]:
import time
free.cache_clear()
for l, r in [(1, 10**9 - 1), (123456789, 987654321), (10**8, 10**9 - 1)]:
    t = time.time(); a = Solution().beautifulNumbers(l, r); t1 = time.time() - t
    assert a == count_fast(l, r)
    print(f"[{l}, {r}]: {a} | {t1:.2f}s")

[1, 999999999]: 670349658 | 0.05s
[123456789, 987654321]: 581119317 | 0.04s
[100000000, 999999999]: 607393752 | 0.04s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi chữ số 0 làm "hỏng" số (bỏ qua các số có 0)

In [15]:
def wrong_skip_zero(l, r):
    return sum(1 for x in range(l, r + 1) if '0' not in str(x) and beautiful(x))

print('sai :', wrong_skip_zero(10, 20), '| đúng:', Solution().beautifulNumbers(10, 20))

sai : 0 | đúng: 2


**Vì sao sai:** tích bằng 0 chia hết cho mọi tổng dương ⇒ số có chữ số 0 **luôn** đẹp.

### Lỗi 2 — Coi số 0 ở đầu (padding) là chữ số thật

In [16]:
def F_leading_zero_bug(N):
    D = list(map(int, str(N)))
    m = len(D)

    @lru_cache(maxsize=None)
    def go(i, s, p, tight):                 # SAI: không có cờ "đã bắt đầu"
        if i == m:
            return 1 if s > 0 and p % s == 0 else 0
        hi = D[i] if tight else 9
        return sum(go(i + 1, s + d, p * d, tight and d == hi) for d in range(hi + 1))

    return go(0, 0, 1, True)

print('sai :', F_leading_zero_bug(100), '| đúng:', Solution().beautifulNumbers(1, 100))
print('ví dụ: 11 bị xem là "011" -> tích 0 -> "đẹp", nhưng thật ra 11 đẹp?', beautiful(11))

sai : 100 | đúng: 25
ví dụ: 11 bị xem là "011" -> tích 0 -> "đẹp", nhưng thật ra 11 đẹp? False


**Vì sao sai:** khi đệm `11` thành `011` để cùng độ dài với `N = 100`, chữ số 0 giả làm tích bằng 0 ⇒ mọi số ngắn hơn `N` đều bị coi là đẹp. Phải phân biệt "chưa bắt đầu" (cờ `started`, hoặc tách riêng nhóm số ít chữ số như cách 3) — điều này cũng đảm bảo tổng > 0, không bao giờ gặp `0 % 0`.

### Lỗi 3 — `F(r) − F(l)` thay vì `F(r) − F(l − 1)`

In [17]:
print('sai :', F(20) - F(10), '| đúng:', F(20) - F(9))

sai : 1 | đúng: 2


**Vì sao sai:** mất chính `l` nếu `l` đẹp (ví dụ `l = 10`).

### Lỗi 4 — Ghi nhớ theo `(vị trí, tổng)` mà bỏ tích

In [18]:
def F_drop_product(N):
    D = list(map(int, str(N)))
    m = len(D)
    memo = {}
    def go(i, s, p, tight, started):
        key = (i, s, tight, started)                # SAI: thiếu p
        if not tight and key in memo:
            return memo[key]
        if i == m:
            return 1 if started and p % s == 0 else 0
        hi = D[i] if tight else 9
        res = 0
        for d in range(hi + 1):
            if not started and d == 0:
                res += go(i + 1, 0, 1, tight and d == hi, False)
            else:
                res += go(i + 1, s + d, p * d, tight and d == hi, True)
        if not tight:
            memo[key] = res
        return res
    return go(0, 0, 1, True, False)

print('sai :', F_drop_product(500), '| đúng:', F(500))

sai : 100 | đúng: 158


**Vì sao sai:** hai tiền tố cùng tổng nhưng khác tích có tương lai khác nhau.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số trong `[l, r]` theo tính chất chữ số" ⇒ digit DP + `F(r) − F(l−1)`.
- Trạng thái có **tích** chữ số ⇒ số tích khác nhau nhỏ (chỉ thừa số 2, 3, 5, 7); có thể biểu diễn bằng bộ số mũ.
- Trường hợp "hấp thụ" (tích 0) ⇒ đếm thẳng `10^{rem}`.
- Hàm `free` không phụ thuộc `N` ⇒ ghi nhớ dùng chung.

## 15. Biến thể của bài

1. Đếm số mà **tổng chia hết cho tích** (tích khác 0).
2. Đếm số mà tích chữ số là **số chính phương** ⇒ trạng thái là **tính chẵn lẻ** các số mũ (16 trạng thái).
3. Đếm số có tổng chữ số chia hết cho `k` ⇒ digit DP theo số dư.
4. Bài liên quan: LC 2376, LC 357, LC 1012.

In [19]:
# Biến thể 2: tích chữ số là số chính phương (tích 0 coi là chính phương)
@lru_cache(maxsize=None)
def free_sq(rem, mask, zero):
    if zero:
        return 10 ** rem
    if rem == 0:
        return 1 if mask == 0 else 0
    PAR = {1: 0, 2: 1, 3: 2, 4: 0, 5: 4, 6: 3, 7: 8, 8: 1, 9: 0}
    total = free_sq(rem - 1, mask, True)
    for d in range(1, 10):
        total += free_sq(rem - 1, mask ^ PAR[d], False)
    return total

def count_square_product(N):
    import math
    return sum(1 for x in range(1, N + 1) if math.isqrt(digit_ps(x)[0]) ** 2 == digit_ps(x)[0])

PAR = {1: 0, 2: 1, 3: 2, 4: 0, 5: 4, 6: 3, 7: 8, 8: 1, 9: 0}
total_2digit = sum(free_sq(1, PAR[d], False) for d in range(1, 10))
print('số 2 chữ số có tích chính phương:', total_2digit, '| vét cạn:', count_square_product(99) - count_square_product(9))

số 2 chữ số có tích chính phương: 26 | vét cạn: 26


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Duyệt | `O((r−l) m)` | Oracle |
| 2 | Khuôn digit DP | nhiều trạng thái | Dễ viết |
| 3 | `free` + tích 0 | ít trạng thái | **Nộp bài** |
| 4 | Bộ số mũ | như 3 | Hiểu cấu trúc tích |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `beautifulNumbers(20, 30)`.

**Bài 2 (tính tay):** `beautifulNumbers(100, 110)`.

**Bài 3 (điền TODO):** viết `count_with_zero(N)` — số trong `[1, N]` có ít nhất một chữ số 0.

**Bài 4 (tự viết lại):** biến thể 1 — tổng chia hết cho tích (tích khác 0).

**Bài 5 (mini challenge):** biến thể 3 — tổng chữ số chia hết cho `k`.

In [20]:
def count_with_zero_todo(N):
    # TODO
    raise NotImplementedError

def sum_divisible_by_product_todo(l, r):
    # TODO
    raise NotImplementedError

def digit_sum_mod_k_todo(l, r, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `20`, `30` có chữ số 0. (2) `22`: `4 % 4 = 0` ✓; `21..29` khác: kiểm tra. (3) Dùng code kiểm tra.

**Bài 2.** (1) Nhìn chữ số hàng chục/đơn vị của các số `100..110`. (2) Cả 11 số từ 100 đến 110 đều chứa chữ số 0. (3) Đáp án `11`.

**Bài 3.** (1) Đếm phần bù: số không có 0. (2) Số không có 0 với `L` chữ số: `9^L`, phần bám sát tương tự digit DP. (3) Hoặc dùng `free` với tích "có 0".

**Bài 4.** (1) Đổi điều kiện cuối thành `p != 0 and s % p == 0`. (2) Tích 0 ⇒ không đếm. (3) Vét cạn.

**Bài 5.** (1) Trạng thái `(rem, s mod k)`. (2) `F(r) − F(l−1)`. (3) Vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
a1 = Solution().beautifulNumbers(20, 30)
a2 = Solution().beautifulNumbers(100, 110)
assert a1 == count_brute(20, 30) and a2 == 11 == count_brute(100, 110)
print('Bài 1:', a1, [x for x in range(20, 31) if beautiful(x)], '| Bài 2:', a2)

Bài 1: 3 [20, 22, 30] | Bài 2: 11


In [22]:
# Bài 3, 4, 5 — dùng một khuôn digit DP tổng quát với hàm "kết thúc" tùy chọn
def digit_count(N, init, step, accept, absorb=None):
    """Đếm x trong [1, N]: trạng thái bắt đầu init, cập nhật step(state, d), chấp nhận accept(state)."""
    if N <= 0:
        return 0
    D = list(map(int, str(N)))
    m = len(D)

    @lru_cache(maxsize=None)
    def free_(rem, st):
        if rem == 0:
            return 1 if accept(st) else 0
        return sum(free_(rem - 1, step(st, d)) for d in range(10))

    total = 0
    for L in range(1, m):
        for d in range(1, 10):
            total += free_(L - 1, step(init, d))
    st = init
    for i, di in enumerate(D):
        for d in range(1 if i == 0 else 0, di):
            total += free_(m - i - 1, step(st, d))
        st = step(st, di)
    return total + (1 if accept(st) else 0)

count_with_zero = lambda N: digit_count(N, False, lambda z, d: z or d == 0, lambda z: z)
sum_div_prod = lambda N: digit_count(N, (0, 1), lambda t, d: (t[0] + d, t[1] * d),
                                     lambda t: t[1] != 0 and t[0] % t[1] == 0)
sum_mod_k = lambda N, k: digit_count(N, 0, lambda s, d: (s + d) % k, lambda s: s == 0)

assert count_with_zero(1000) == sum(1 for x in range(1, 1001) if '0' in str(x))
assert sum_div_prod(5000) == sum(1 for x in range(1, 5001) if digit_ps(x)[0] and digit_ps(x)[1] % digit_ps(x)[0] == 0)
assert sum_mod_k(5000, 7) == sum(1 for x in range(1, 5001) if sum(map(int, str(x))) % 7 == 0)
print('Bài 3:', count_with_zero(1000), '| Bài 4:', sum_div_prod(5000), '| Bài 5:', sum_mod_k(5000, 7))

Bài 3: 181 | Bài 4: 38 | Bài 5: 712


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
free(rem, s, p): p == 0 -> 10**rem; rem == 0 -> [p % s == 0]; else Σ_d free(rem-1, s+d, p*d)
F(N): Σ_{L<m} Σ_{d=1..9} free(L-1, d, d)
      + Σ_i Σ_{d < D[i]} free(m-i-1, s+d, p*d)   (d ≥ 1 ở vị trí đầu)
      + [N đẹp]
đáp án = F(r) - F(l-1)
```

**Checklist:**

- [ ] Số có chữ số 0 luôn đẹp?
- [ ] Không đếm số 0 / không chia cho tổng 0?
- [ ] Trạng thái giữ cả **tổng** và **tích**?
- [ ] `F(r) − F(l − 1)`?
- [ ] Ghi nhớ dùng chung giữa hai lần gọi `F`?